# Run Energy Price Calculation Script

This notebook executes the `bereken_energie_prijzen.py` script to calculate energy prices for Q3 2025 - Q2 2026.

## setup

In [26]:
#!/usr/bin/env python3
import os
from pathlib import Path
import time
import pandas as pd

# Define parameters
BASE_DIR = Path(os.getcwd()).resolve().parent
INPUT_CSV = BASE_DIR / "energieverbruik" / "P1e-2025-7-1-2026-7-1.csv"
PREPROCESSED_CSV = BASE_DIR / "energieverbruik" / "P1e-preprocessed.csv"
OUTPUT_CSV = BASE_DIR / "energieverbruik" / "P1e-processed.csv"
START_DATE = pd.Timestamp("2025-07-01 00:00:00")
END_DATE = pd.Timestamp("2026-06-30 23:59:59")

print(f"Parameters:")
print(f"  Bron: {INPUT_CSV}")
print(f"  Start date: {START_DATE}")
print(f"  End date: {END_DATE}")
print(f"  Preprocessed: {PREPROCESSED_CSV}")
print(f"  result: {OUTPUT_CSV}")

Parameters:
  Bron: /Users/felixdonkers/Library/Mobile Documents/com~apple~CloudDocs/my_python_projects/misc_projects/thuisbatterij/energieverbruik/P1e-2025-7-1-2026-7-1.csv
  Start date: 2025-07-01 00:00:00
  End date: 2026-06-30 23:59:59
  Preprocessed: /Users/felixdonkers/Library/Mobile Documents/com~apple~CloudDocs/my_python_projects/misc_projects/thuisbatterij/energieverbruik/P1e-preprocessed.csv
  result: /Users/felixdonkers/Library/Mobile Documents/com~apple~CloudDocs/my_python_projects/misc_projects/thuisbatterij/energieverbruik/P1e-processed.csv


## pre-process data
convert to table with rows per 15 min

In [18]:
def read_source_csv(path: Path) -> pd.DataFrame:
    columns = [
        "datum_nl", 
        "Import T1 kWh", 
        "Import T2 kWh", 
        "Export T1 kWh", 
        "Export T2 kWh", 
        "L1 max W", 
        "L2 max W", 
        "L3 max W"
        ]
    df = pd.read_csv(
        path,
        sep=",",
        decimal=".",
        skiprows=1,
        header=None,
        names=columns,
        encoding="utf-8",
    )

    # drop unnecessary columns
    df = df.drop(["L1 max W","L2 max W","L3 max W"], axis=1)

    # filter on date range
    df["datum_nl"] = pd.to_datetime(df["datum_nl"], format="%Y-%m-%d %H:%M", errors="coerce")
    df = df[df["datum_nl"] >= START_DATE].copy()
    df = df[df["datum_nl"] <= END_DATE].copy()

    # calculate energy usage per 15-minute interval from cumulative meter readings
    df["total_import_kWh"] = df["Import T1 kWh"] + df["Import T2 kWh"]
    df["total_export_kWh"] = df["Export T1 kWh"] + df["Export T2 kWh"]
    df["netto_usage_kWh"] = df["total_import_kWh"] - df["total_export_kWh"]
    df["energy_usage_kWh_per_15m"] = (
        df["netto_usage_kWh"].diff().fillna(0.0)
    )

    # drop unnecessary columns
    df = df.drop(["Import T1 kWh","Import T2 kWh"], axis=1)
    df = df.drop(["Export T1 kWh","Export T2 kWh"], axis=1)
    df.to_csv(PREPROCESSED_CSV, index=False, lineterminator="\n", float_format='%.3f')
    return df

preprocessed = read_source_csv(INPUT_CSV)
print(f"Rows in preprocessed: {len(preprocessed)}, elements per row: {len(preprocessed.columns)}")

print(preprocessed.head(5))
print(".....")
print(preprocessed.tail(5))

Rows in preprocessed: 35040, elements per row: 5
             datum_nl  total_import_kWh  total_export_kWh  netto_usage_kWh  \
0 2025-07-01 00:00:00         23095.908               0.0        23095.908   
1 2025-07-01 00:15:00         23095.952               0.0        23095.952   
2 2025-07-01 00:30:00         23096.004               0.0        23096.004   
3 2025-07-01 00:45:00         23096.049               0.0        23096.049   
4 2025-07-01 01:00:00         23096.094               0.0        23096.094   

   energy_usage_kWh_per_15m  
0                     0.000  
1                     0.044  
2                     0.052  
3                     0.045  
4                     0.045  
.....
                 datum_nl  total_import_kWh  total_export_kWh  \
35035 2026-06-30 22:45:00         27687.320               0.0   
35036 2026-06-30 23:00:00         27687.387               0.0   
35037 2026-06-30 23:15:00         27687.445               0.0   
35038 2026-06-30 23:30:00         27

## post-process
convert to table with rows per day

In [28]:
def format_value(value: float) -> str:
    return f"{value:.3f}"

def postprocess_energy_usage(df: pd.DataFrame) -> None:
    time_slots = [f"{hour:02d}:{minute:02d}" for hour in range(24) for minute in (0, 15, 30, 45)]
    header = ["date", *time_slots, "total", "minimum", "average", "maximum"]

    rows = []
    for day, day_df in df.groupby(df["datum_nl"].dt.normalize()):
        ordered = day_df.sort_values("datum_nl")
        values = ordered["energy_usage_kWh_per_15m"].tolist()
        if len(values) != len(time_slots):
            values = values[: len(time_slots)]
            if len(values) < len(time_slots):
                # Fill missing values with 0.0 for any missing time slots (summer time or other gaps)
                values.extend([float(0)] * (len(time_slots) - len(values)))

        total = float(sum(values))
        minimum = float(min(values))
        maximum = float(max(values))
        average = total / len(values)

        row = [day.strftime("%Y-%m-%d")]
        row.extend(format_value(value) if pd.notna(value) else "" for value in values)
        row.extend([
            format_value(total),
            format_value(minimum),
            format_value(average),
            format_value(maximum),
        ])
        rows.append(row)

    output_df = pd.DataFrame(rows, columns=header)
    output_df.to_csv(OUTPUT_CSV, index=False, lineterminator="\n")

    print(f"Wrote {len(rows)} rows to {OUTPUT_CSV}")

postprocess_energy_usage(preprocessed)

Wrote 365 rows to /Users/felixdonkers/Library/Mobile Documents/com~apple~CloudDocs/my_python_projects/misc_projects/thuisbatterij/energieverbruik/P1e-processed.csv
